In [223]:
import sys
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re
from datetime import datetime
import time
import json


In [224]:
my_dict={"User-Agent":"Prudhvi Raj, vicky.prudhvi@gmail.com"}
URL='https://www.federalreserve.gov/newsevents/pressreleases/monetary20240131a.htm'


In [225]:
clean_paragraphs =[]
paragraph=body.find_all("p")
for p in paragraph:
    text = p.get_text(" ",strip=True)
    if text.startswith("For media inquiries"):
        break
    clean_paragraphs.append(text)
    
        

In [226]:
def get_statement(url):
   
    match = re.search(r"monetary(\d{8})", url)
    raw = match.group(1)   
    date=datetime.strptime(raw, "%Y%m%d").date().isoformat()
    # print(date)
    responds= requests.get(url, headers=my_dict)
    responds.encoding = "utf-8"
    if responds.status_code!=200:
        print("failed to pull data")
        return None
    soup = BeautifulSoup(responds.text, "html.parser")
    body = soup.select_one("#article div.col-md-8:not(.heading)")
    if body is None:
       print("no statement body found:", url)
       return None
    clean_paragraphs =[]
    paragraph=body.find_all("p")
    for p in paragraph:        
        text = p.get_text(" ",strip=True)
        if text.startswith(("For media inquiries","Implementation Note")):
            break
        if not text:
            continue
        clean_paragraphs.append(text)
    statement_text = "\n\n".join(clean_paragraphs)
    return {"url":url,"date":date,"doc_type":"statement","text":statement_text}
    
    

In [227]:
formcal='https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm'
respond=requests.get(formcal,headers=my_dict)
respond.status_code

200

In [228]:
soup=BeautifulSoup(respond.text,"html.parser")
statement_urls = set()
aclass=soup.find_all("a")
for a in aclass:
    link=a.get("href")
    if link is None:
        continue
    # print(link)
    raw_data=re.search(r"monetary\d{8}a\.htm", link)
    if raw_data is None:
        continue
    statement_urls.add(f'https://www.federalreserve.gov{link}')

statement_urls=sorted(statement_urls)

In [229]:
statements=[]
for s in statement_urls:
    result = get_statement(s)
    if result is not None:
        statements.append(result)
    time.sleep(1)

In [230]:
for s in statements:
    if s['date']=="2025-08-22":
        s["doc_type"] = 'framework'

In [231]:
os.makedirs("data/raw", exist_ok=True)
with open("data/raw/statements.json","w",encoding="utf-8") as f:
    json.dump(statements,f,ensure_ascii=False, indent=2)